# Nepali Sentiment Analysis

**Project:** End-to-end machine learning pipeline for Nepali-language sentiment classification.

**Sections:**
1. Problem Definition
2. Data Loading, Cleaning & EDA
3. Text Preprocessing (normalization, stopwords, TF-IDF)
4. Train/Test Split
5. Model Training (Naive Bayes, Logistic Regression, Linear SVM)
6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. Results Summary & Model Saving

## 1. Problem Definition

- **Dataset:** Nepali text sentiment dataset stored locally in `data/raw/nepali_sentiment_raw.csv`.
- **Task:** Binary or multi-class text classification, depending on the labels present in the dataset.
- **Target variable:** `label`, normalized from common sentiment column names such as `label`, `sentiment`, `class`, or `target`.
- **Objective:** Predict the sentiment of a Nepali text sample using classical machine learning models and TF-IDF features.

In [ ]:
from pathlib import Path
import os
import warnings
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
MPL_CONFIG_DIR = PROJECT_ROOT / '.matplotlib'
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG_DIR))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'nepali_sentiment_raw.csv'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'nepali_sentiment_clean.csv'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'

for directory in [FIGURES_DIR, RESULTS_DIR, MODELS_DIR, PROCESSED_DATA_PATH.parent]:
    directory.mkdir(parents=True, exist_ok=True)

## 2. Data Loading, Cleaning & EDA

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f'Raw dataset not found at {RAW_DATA_PATH}. Download the CSV and place it in data/raw/.'
    )

df = pd.read_csv(RAW_DATA_PATH)
df.head()

In [ ]:
# Basic inspection
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
def find_column(columns, candidates):
    normalized = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    return None

text_col = find_column(df.columns, ['text', 'sentence', 'review', 'comment', 'content', 'tweet', 'message'])
label_col = find_column(df.columns, ['label', 'sentiment', 'class', 'target', 'category'])

if text_col is None or label_col is None:
    raise ValueError(
        'Could not identify text and label columns automatically. '
        f'Available columns: {list(df.columns)}'
    )

df = df.rename(columns={text_col: 'text', label_col: 'label'})
df = df[['text', 'label']].copy()
df['label'] = df['label'].astype(str).str.strip().str.lower()
print(f'Using text column: {text_col}')
print(f'Using label column: {label_col}')
df.head()

In [ ]:
# Drop duplicates and missing values
df = df.drop_duplicates()
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].astype(str).str.strip().ne('')]
df.shape

In [ ]:
# Class balance
ax = df['label'].value_counts().plot(kind='bar', title='Sentiment Class Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sentiment_distribution.png', dpi=150)
plt.show()

In [ ]:
# Text length distribution
df['text_length'] = df['text'].astype(str).apply(len)
df['text_length'].hist(bins=40)
plt.title('Text Length Distribution')
plt.xlabel('Character count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'text_length_distribution.png', dpi=150)
plt.show()

## 3. Text Preprocessing

Steps: Unicode normalization, removing punctuation/numerals, Nepali stopword removal, TF-IDF vectorization.

In [ ]:
import unicodedata

def clean_text(text):
    text = str(text)
    text = unicodedata.normalize('NFC', text)  # normalize Devanagari encoding variants
    text = re.sub(r'[^\u0900-\u097F\s]', ' ', text)  # keep only Devanagari + whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].ne('')].copy()
df[['text', 'clean_text']].head()

In [ ]:
nepali_stopwords = {
    'अनि', 'अथवा', 'अझै', 'आज', 'त्यसैले', 'त्यसको', 'त्यसमा', 'त्यो', 'त', 'तर',
    'तथा', 'तिमी', 'तिम्रो', 'तिनले', 'ती', 'थिए', 'थियो', 'छ', 'छु', 'छन्', 'छैन',
    'जब', 'जसले', 'जसमा', 'जुन', 'जे', 'जो', 'न', 'नि', 'पनि', 'पर्‍यो', 'पहिले',
    'भए', 'भएको', 'भने', 'भन्ने', 'भित्र', 'म', 'मा', 'मात्र', 'मेरो', 'यति',
    'यदि', 'यस', 'यसका', 'यसको', 'यसले', 'यहाँ', 'या', 'र', 'रही', 'रहेका',
    'लिए', 'लाई', 'ले', 'वा', 'संग', 'सँग', 'सबै', 'हो', 'हुन', 'हुन्छ', 'हुन्'
}

def remove_stopwords(text):
    return ' '.join(word for word in text.split() if word not in nepali_stopwords)

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
df = df[df['clean_text'].ne('')].copy()
df[['text', 'clean_text']].head()

In [ ]:
# Save processed data
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f'Saved processed data to {PROCESSED_DATA_PATH}')

## 4. Train/Test Split & TF-IDF Vectorization

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

label_counts = df['label'].value_counts()
stratify_labels = df['label'] if label_counts.min() >= 2 else None
if stratify_labels is None:
    print('Warning: at least one class has fewer than 2 samples; using an unstratified split.')

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=stratify_labels
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    token_pattern=r'(?u)[\u0900-\u097F]+',
)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train.shape, X_test.shape

## 5. Model Training

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.base import clone

models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Linear SVM': LinearSVC(class_weight='balanced')
}

for name, model in models.items():
    model.fit(X_train, y_train)

## 6. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

results = []
for name, model in models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average='macro', zero_division=0
    )
    results.append({'model': name, 'accuracy': acc, 'precision': precision, 'recall': recall, 'f1_macro': f1})
    print(f'--- {name} ---')
    print(classification_report(y_test, preds, zero_division=0))

results_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
results_df

In [ ]:
results_path = RESULTS_DIR / 'model_comparison.csv'
results_df.to_csv(results_path, index=False)
print(f'Saved model comparison to {results_path}')

In [ ]:
# Confusion matrix for the best model
best_model_name = results_df.sort_values('f1_macro', ascending=False).iloc[0]['model']
best_model = models[best_model_name]
preds = best_model.predict(X_test)

labels = list(best_model.classes_)
cm = confusion_matrix(y_test, preds, labels=labels)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

## 7. Results Summary & Model Saving

In [ ]:
final_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    token_pattern=r'(?u)[\u0900-\u097F]+',
)
X_all = final_vectorizer.fit_transform(df['clean_text'])
final_model = clone(best_model)
final_model.fit(X_all, df['label'])

model_artifact = {
    'model': final_model,
    'vectorizer': final_vectorizer,
    'best_model_name': best_model_name,
    'metrics': results_df.to_dict(orient='records'),
    'stopwords': sorted(nepali_stopwords),
}

model_path = MODELS_DIR / 'final_model.pkl'
joblib.dump(model_artifact, model_path)
print(f'Saved best model ({best_model_name}) and vectorizer to {model_path}')

In [ ]:
def predict_sentiment(text):
    cleaned = remove_stopwords(clean_text(text))
    features = final_vectorizer.transform([cleaned])
    return final_model.predict(features)[0]

# Example usage after training:
# predict_sentiment('यो फिल्म निकै राम्रो छ')

### Summary

- Best performing model: selected automatically by highest macro F1 score in `results_df`.
- Key takeaways: TF-IDF features with linear models provide a strong, interpretable baseline for Nepali sentiment classification.
- Limitations: performance depends on dataset quality, class balance, spelling variation, informal Nepali, and mixed Devanagari/Romanized text.